# 7LBF 신약 후보 발굴 사례 · v001

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/demo.ipynb)

배지는 GitHub `main`의 최신 노트북을 엽니다. 노트북을 수정해 올릴 때는 이 제목과 설정 셀의 버전을 함께 올립니다.

이 데모는 논문에서 실험한 **PDGFRαbp-7LBF 한 후보**를 표적 설정부터 결합과 세포 신호 억제까지 추적합니다. 모델 생성과 후보 선별은 논문의 공개 기록을 따라갑니다. Colab은 공개 실험 원자료를 읽어 결과를 확인하고 ZIP으로 저장합니다. 새 후보 생성이나 실험 재현은 포함되어 있지 않습니다.

[2026년 논문과 원자료](https://pmc.ncbi.nlm.nih.gov/articles/PMC12852815/)

## 1. 표적에서 후보까지

`사람 PDGFRα → 결합할 단백질 뼈대 생성 → 아미노산 서열 설계 → 계산 선별 → 결합 및 세포 실험`

연구팀은 구조 참조 번호 **PDB 7LBF**를 바탕으로 사람 PDGFRα에 결합할 단백질을 설계했습니다. RFdiffusion으로 뼈대, ProteinMPNN으로 서열을 만들고, AlphaFold 2와 Rosetta 지표로 계산 선별했습니다. 이어 5,427개 설계의 라이브러리를 실험으로 선별해 **PDGFRαbp-7LBF**를 얻었습니다. PDB 번호와 설계 후보 이름을 구분해서 읽으면 됩니다. [논문 설계 과정과 그림 2](https://pmc.ncbi.nlm.nih.gov/articles/PMC12852815/)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

from google.colab import files
from IPython.display import Image, display
import pandas as pd

VERSION = "v001"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
BASE = Path("/content") / f"run_7lbf_{RUN_ID}"
REPO = BASE / "repo"
OUTPUT = BASE / "7lbf_results"
BASE.mkdir()

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/HisameOgasahara/tmp_protein.git", str(REPO)
], check=True)
COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print(f"Notebook {VERSION} · Script commit {COMMIT}")

## 2. 같은 후보의 실험값 보기

논문의 그림 5e 원자료에서 농도별 신호 측정값을 읽습니다. 점은 반복 측정값이고, 선은 해당 농도에서 기록된 측정값의 평균입니다. K_D는 표적에 얼마나 잘 결합하는지, IC50은 실험에서 신호를 절반으로 줄이는 데 얼마나 필요한지를 나타냅니다. [논문 그림 2·5](https://pmc.ncbi.nlm.nih.gov/articles/PMC12852815/)

In [ ]:
subprocess.run([
    sys.executable, str(REPO / "scripts" / "build_7lbf_report.py"),
    "--output", str(OUTPUT), "--version", VERSION, "--commit", COMMIT,
], check=True)

display(pd.read_csv(OUTPUT / "data" / "candidate_path.csv"))
display(pd.read_csv(OUTPUT / "data" / "reported_endpoints.csv"))
display(Image(filename=str(OUTPUT / "figures" / "7lbf_inhibition.png")))
display(Image(filename=str(OUTPUT / "figures" / "7lbf_dose_response.png")))

PDGFRαbp-7LBF는 SPR 결합 실험에서 **137 pM**의 `K_D`를 보였습니다. 같은 후보가 세포에서 ERK와 AKT의 인산화를 줄였고, 논문이 보고한 `IC50`은 각각 **19.6 nM**, **38.25 nM**였습니다. 앞서 진행한 단백질 설계 데모의 RMSD는 목표 구조의 재현성을 보여주었고, 여기서 확인한 두 실험은 **표적 결합**과 **세포 신호 변화**를 각각 보여줍니다. [논문 그림 2·5](https://pmc.ncbi.nlm.nih.gov/articles/PMC12852815/)

## 3. 결과 ZIP 다운로드

후보 추적표, 논문에 보고된 지표, 그림 5e 원자료 CSV, 그래프와 출처 기록을 저장합니다.

In [ ]:
import shutil

if not (OUTPUT / "provenance" / "sources.json").is_file():
    raise RuntimeError("먼저 실험값을 읽는 셀을 완료하세요.")
archive = shutil.make_archive(str(BASE / f"7lbf_{VERSION}_results"), "zip", root_dir=OUTPUT)
files.download(archive)